# SI: convergence of the initial $\dot\lambda$

Error in the initial $\dot\lambda$ vs. $\ln d/d$, for $u=0.1$ and $u=0.7$.

Data comes from `extremal_mil.gradient_convergence`:

```
OMP_NUM_THREADS=1 python -m extremal_mil.gradient_convergence
```

Writes `figures/si_gradient_convergence.pdf`.

In [ ]:
# NEEDS HUMAN REVIEW
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from extremal_mil import gradient_convergence as gc
from extremal_mil import plotting

plt.rcParams.update(plotting.RCPARAMS)
%config InlineBackend.figure_format = 'svg'

FIG_DIR = Path.cwd().parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

D = gc.available()
print(f"d = {D}")
print(f"lambda0={gc.LAM0}  rho0={gc.RHO0}  alpha={gc.ALPHA}  "
      f"rho_true={gc.RHO_TRUE}  q={gc.Q}  ell={gc.ELL}")
for u in gc.U_VALS:
    print(f"  theory lambda-dot at u={u}: {gc.theory_lam(u):+.4f}")

In [ ]:
cells = {d: gc.load(d) for d in D}
x = np.array([np.log(d) / d for d in D])

fig, axes = plt.subplots(1, 2, figsize=(plotting.TEXT_W, 2.6),
                         layout="constrained")

for j, (ax, u) in enumerate(zip(axes, gc.U_VALS)):
    draws = [cells[d]["err"][j] for d in D]
    m = np.array([v.mean() for v in draws])
    sem = np.array([v.std(ddof=1) / np.sqrt(len(v)) for v in draws])

    for xi, v in zip(x, draws):
        ax.plot(np.full(len(v), xi), v, ".", color="0.72", ms=3, zorder=2)
    ax.errorbar(x, m, yerr=sem, fmt="o", color="k", ms=4.5, capsize=2.5,
                lw=1.0, zorder=5)
    # guide through the origin and the smallest-d point; nothing is fitted
    g = np.linspace(0, x.max() * 1.06, 50)
    ax.plot(g, m[0] / x[0] * g, "--", color="0.45", lw=1.1, zorder=3)

    ax.set_xlabel(r"$\ln d\,/\,d$")
    ax.set_title(rf"$u={u}$", pad=4)
    ax.set_xlim(0, x.max() * 1.06)
    ax.set_ylim(0, None)
    sec = ax.secondary_xaxis("top")
    show = [d for d in (22, 46, 100, 150, 270) if d in D]
    sec.set_xticks([np.log(d) / d for d in show])
    sec.set_xticklabels([str(d) for d in show])
    sec.set_xlabel("$d$", labelpad=2)

axes[0].set_ylabel(r"$|\dot\lambda_{\rm emp}-\dot\lambda_{\rm ODE}|$")
fig.legend(handles=[
    Line2D([], [], color="0.72", marker=".", ls="none", ms=5,
           label="individual datasets"),
    Line2D([], [], color="k", marker="o", ls="none", ms=4.5,
           label=r"mean $\pm$ s.e.m."),
    Line2D([], [], color="0.45", ls="--", lw=1.1, label=r"$\ln d/d$")],
    loc="outside lower center", ncols=3, frameon=False)

fig.savefig(FIG_DIR / "si_gradient_convergence.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# The numbers behind the figure.
print(f"{'d':>5}{'n':>9}{'reps':>6}" + "".join(f"{f'u={u}':>12}" for u in gc.U_VALS))
for d in D:
    c = cells[d]
    row = "".join(f"{c['err'][j].mean():>12.4f}" for j in range(len(gc.U_VALS)))
    print(f"{d:>5}{int(c['n']):>9}{int(c['n_reps']):>6}{row}")